## 变量说明

### 1. FT（植被功能型）
FT 表示采样植被的植被功能型。将所有样本类别聚合到 **0.1°** 后，共得到以下 4 类，属于**类别变量**：

- **Tree**：聚合前像元仅包含 `Tree`、`Small tree`
- **Shrub**：聚合前像元包含 `Shrub`、`Large shrub`、`Subshrub`、`Shrub/Small tree`
- **Herb**：聚合前像元包含 `Grass`、`Forb`、`Sedge`、`Graminoid`、`Geophyte`
- **Mix**：聚合后存在混合值的情况

---

### 2. IGBP 与 LC
- **IGBP**：基于 IGBP 标准的 17 类地物类型
- **LC**：对 IGBP 进一步合并后的地物类型

#### LC 合并规则
- **NF** = `ENF + DNF`
- **BF** = `EBF + DBF`
- **SH** = `OSH + CSH`
- **Herb** = `CRO + SAV + GRA`
- **NonVeg** = `URB + BAR + SNO + Water`
- **MixHerb** = `CVM + WSA`
- **MF** 和 **WET** 保留原类型不变

---

## 后续试验建议

### 方案 1
**FT 方案分类**  
- 固定输入变量集：
  - 3 个季节因子（由纬度和 Day of Year 构成）
  - `6VOD`
  - `LAI`
  - `Hveg`
  - `SM`
  - `LST`
  - `13` 个 IGBP 比例变量（**不再使用非植被类型**）
- 数据集空间分带：**0.5°**
- 划分策略：**训练 : 验证 : 测试 = 8 : 1 : 1**
- 样本预处理：
  - 去除 `lfmc = 0`
  - 去除 `VOD` 全为 `NaN` 的样本
- 代价函数：**MAE**

---

### 方案 2
**IGBP Landcover 原类型分类主地物类型**  
- 固定输入变量集
- **0.5°** 空间分带
- 相同划分策略
- 相同样本预处理
- 代价函数：**MAE**

---

### 方案 3
**IGBP Landcover 原类型分类主地物类型 + 新变量**  
- 新变量：
  - 主地物类型比值
  - 像元混合度信息熵
- 固定输入变量集
- **0.5°** 空间分带
- 相同划分策略
- 相同样本预处理
- 代价函数：**MAE**

---

### 方案 4
**IGBP Landcover 原类型分类主地物类型 + 新变量 + 样本权重**  
- 新变量：
  - 主地物类型比值
  - 像元混合度信息熵
- 样本权重：**主地物类型比值**
- 固定输入变量集
- **0.5°** 空间分带
- 相同划分策略
- 相同样本预处理
- 代价函数：**MAE**

---

### 方案 5
**LC Landcover 主地物类型 + 新变量 + 样本权重**  
- 新变量：
  - 主地物类型比值
  - 像元混合度信息熵
- 样本权重：**主地物类型比值**
- 固定输入变量集
- **0.5°** 空间分带
- 相同划分策略
- 相同样本预处理
- 代价函数：**MAE**

---

### 方案 6
**LC Landcover 主地物类型 + 新变量 + 样本权重 + 一致性筛选**  
- 新变量：
  - 主地物类型比值
  - 像元混合度信息熵
- 样本权重：**主地物类型比值**
- 固定输入变量集
- **0.5°** 空间分带
- 相同划分策略
- 样本预处理新增：
  - 样本 IGBP 与像元主类型一致性筛选
- 代价函数：**MAE**

---

### 方案 7
**LC Landcover 主地物类型 + 新变量 + 样本权重 + 双重筛选**  
- 新变量：
  - 主地物类型比值
  - 像元混合度信息熵
- 样本权重：**主地物类型比值**
- 固定输入变量集
- **0.5°** 空间分带
- 相同划分策略
- 样本预处理新增：
  - 一致性筛选
  - 非植被占比筛选
- 代价函数：**MAE**

# 每个方案运行5个模型：CatBoost、XGBoost、LGBM、MLP、TabTransformer，每个模型运行3遍，记录平均精度

# 1 按照上述的初步组织，实现以上内容的模型——机器学习部分——Project312

In [1]:
# 先进行检查
import sys
from pathlib import Path

project_root = Path(r"D:\Python\jupyter\jupyter\LFMCRegressor")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from function.lfmc_ml.lfmc_batch_common import load_base_dataframe

df, igbp_num_cols = load_base_dataframe()

print("lc_dom_frac min/max:", df["lc_dom_frac"].min(), df["lc_dom_frac"].max())
print("igbp_dom_frac min/max:", df["igbp_dom_frac"].min(), df["igbp_dom_frac"].max())

print("lc_weight min/max:", df["lc_weight"].min(), df["lc_weight"].max())
print("igbp_weight min/max:", df["igbp_weight"].min(), df["igbp_weight"].max())

print("Negative lc_weight:", (df["lc_weight"] < 0).sum())
print("Negative igbp_weight:", (df["igbp_weight"] < 0).sum())


lc_dom_frac min/max: 0.2625 1.0
igbp_dom_frac min/max: 0.42857143 1.0
lc_weight min/max: 0.2 1.0
igbp_weight min/max: 0.2 1.0
Negative lc_weight: 0
Negative igbp_weight: 0


In [2]:
import sys
from pathlib import Path

project_root = Path(r"D:\Python\jupyter\jupyter\LFMCRegressor")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from function.lfmc_ml.lfmc_batch_common import (
    load_base_dataframe,
    build_scheme_frame,
    SCHEMES,
)

df, igbp_num_cols = load_base_dataframe()

for scheme in SCHEMES:
    scheme_df, num_cols, cat_col = build_scheme_frame(df, scheme, igbp_num_cols)
    w = scheme_df["sample_weight"]

    print(f"\n=== {scheme.code} ===")
    print("rows:", len(scheme_df))
    print("weight min/max:", w.min(), w.max())
    print("negative weights:", (w < 0).sum())
    print("nan weights:", w.isna().sum())



=== S1_FT ===
rows: 47200
weight min/max: 1.0 1.0
negative weights: 0
nan weights: 0

=== S2_IGBP_DOM ===
rows: 47200
weight min/max: 1.0 1.0
negative weights: 0
nan weights: 0

=== S3_IGBP_DOM_FE ===
rows: 47191
weight min/max: 1.0 1.0
negative weights: 0
nan weights: 0

=== S4_IGBP_DOM_FE_W ===
rows: 47191
weight min/max: 0.2 1.0
negative weights: 0
nan weights: 0

=== S5_LC_DOM_FE_W ===
rows: 47200
weight min/max: 0.2 1.0
negative weights: 0
nan weights: 0

=== S6_LC_DOM_FE_W_CONS ===
rows: 34667
weight min/max: 0.2 1.0
negative weights: 0
nan weights: 0

=== S7_LC_DOM_FE_W_CONS_NV ===
rows: 32352
weight min/max: 0.2 1.0
negative weights: 0
nan weights: 0


In [3]:
!python ..\function\lfmc_ml\run_lfmc_ml_batch.py --repeats 3

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001013 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3328
[LightGBM] [Info] Number of data points in the train set: 38435, number of used features: 25
[LightGBM] [Info] Start training from score 105.500000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

# 2 按照上述的初步组织，实现以上内容的模型——深度学习部分——tau_omega_env

In [2]:
!python ..\function\lfmc_ml\run_lfmc_dl_batch.py --repeats 3

Device: cuda

=== Running S1_FT | FT baseline ===
  [S1_FT] MLP repeat 1/3
  [S1_FT] TabTransformer repeat 1/3
  [S1_FT] MLP repeat 2/3
  [S1_FT] TabTransformer repeat 2/3
  [S1_FT] MLP repeat 3/3
  [S1_FT] TabTransformer repeat 3/3

=== Running S2_IGBP_DOM | dominant IGBP class ===
  [S2_IGBP_DOM] MLP repeat 1/3
  [S2_IGBP_DOM] TabTransformer repeat 1/3
  [S2_IGBP_DOM] MLP repeat 2/3
  [S2_IGBP_DOM] TabTransformer repeat 2/3
  [S2_IGBP_DOM] MLP repeat 3/3
  [S2_IGBP_DOM] TabTransformer repeat 3/3

=== Running S3_IGBP_DOM_FE | dominant IGBP + frac + entropy ===
  [S3_IGBP_DOM_FE] MLP repeat 1/3
  [S3_IGBP_DOM_FE] TabTransformer repeat 1/3
  [S3_IGBP_DOM_FE] MLP repeat 2/3
  [S3_IGBP_DOM_FE] TabTransformer repeat 2/3
  [S3_IGBP_DOM_FE] MLP repeat 3/3
  [S3_IGBP_DOM_FE] TabTransformer repeat 3/3

=== Running S4_IGBP_DOM_FE_W | dominant IGBP + frac + entropy + weight ===
  [S4_IGBP_DOM_FE_W] MLP repeat 1/3
  [S4_IGBP_DOM_FE_W] TabTransformer repeat 1/3
  [S4_IGBP_DOM_FE_W] MLP repeat 2/3


# 3 读取机器学习结果——不限

In [3]:
import pandas as pd
from pathlib import Path

out_dir = Path(r"d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs")

ml_raw = pd.read_csv(out_dir / "lfmc_7schemes_ml_raw.csv")
ml_summary = pd.read_csv(out_dir / "lfmc_7schemes_ml_summary.csv")

print("=== ML Raw ===")
display(ml_raw.head())

print("=== ML Summary ===")
display(ml_summary.sort_values(["scheme", "model", "split"]))


=== ML Raw ===


,scheme,scheme_desc,model,repeat,split,n,MAE,RMSE,R
0,S1_FT,FT baseline,CatBoost,1,train,38435,16.857042,25.276495,0.744628
1,S1_FT,FT baseline,CatBoost,1,val,4821,16.876439,25.585086,0.550016
2,S1_FT,FT baseline,CatBoost,1,test,3944,19.164068,28.074080,0.634839
3,S1_FT,FT baseline,LightGBM,1,train,38435,11.503443,19.685727,0.857675
4,S1_FT,FT baseline,LightGBM,1,val,4821,17.463231,25.562473,0.562562


=== ML Summary ===


,index,scheme,model,split,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R_mean,R_std
0,0,S1_FT,CatBoost,test,19.178567,0.027228,28.095391,0.132122,0.634050,0.004386
1,1,S1_FT,CatBoost,train,17.001684,0.195250,25.436676,0.239795,0.741157,0.005416
2,2,S1_FT,CatBoost,val,16.843010,0.029992,25.487803,0.089818,0.554105,0.003780
3,3,S1_FT,LightGBM,test,20.128022,0.011253,28.524253,0.097157,0.627783,0.002781
4,4,S1_FT,LightGBM,train,11.532010,0.039900,19.708450,0.029744,0.857331,0.000443
...,...,...,...,...,...,...,...,...,...,...
58,58,S7_LC_DOM_FE_W_CONS_NV,LightGBM,train,11.542742,0.109060,19.802129,0.079883,0.855875,0.001474
59,59,S7_LC_DOM_FE_W_CONS_NV,LightGBM,val,21.390190,0.111268,29.995002,0.158251,0.666711,0.004818
60,60,S7_LC_DOM_FE_W_CONS_NV,XGBoost,test,20.204767,0.145405,28.626162,0.138322,0.644609,0.004828
61,61,S7_LC_DOM_FE_W_CONS_NV,XGBoost,train,9.614730,0.139776,17.578033,0.151158,0.891848,0.002318


# 4 读取深度学习结果——不限

In [4]:
import pandas as pd
from pathlib import Path

out_dir = Path(r"d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs")

dl_raw = pd.read_csv(out_dir / "lfmc_7schemes_dl_raw.csv")
dl_summary = pd.read_csv(out_dir / "lfmc_7schemes_dl_summary.csv")

print("=== DL Raw ===")
display(dl_raw.head())

print("=== DL Summary ===")
display(dl_summary.sort_values(["scheme", "model", "split"]))


=== DL Raw ===


,scheme,scheme_desc,model,repeat,split,n,MAE,RMSE,R
0,S1_FT,FT baseline,MLP,1,train,38435,18.961416,27.917930,0.674868
1,S1_FT,FT baseline,MLP,1,val,4821,18.719733,27.885195,0.452071
2,S1_FT,FT baseline,MLP,1,test,3944,21.568049,31.326081,0.536000
3,S1_FT,FT baseline,TabTransformer,1,train,38435,18.880459,27.721862,0.675739
4,S1_FT,FT baseline,TabTransformer,1,val,4821,18.440577,27.422057,0.471387


=== DL Summary ===


,index,scheme,model,split,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R_mean,R_std
0,0,S1_FT,MLP,test,21.594290,0.166087,31.120537,0.245082,0.549268,0.013560
1,1,S1_FT,MLP,train,18.585124,0.379346,27.340582,0.552383,0.687162,0.011814
2,2,S1_FT,MLP,val,18.589887,0.195387,27.677542,0.180638,0.462249,0.008842
3,3,S1_FT,TabTransformer,test,21.675884,0.389958,31.309555,0.534298,0.538416,0.018834
4,4,S1_FT,TabTransformer,train,18.959359,0.165884,27.787284,0.192194,0.673131,0.006903
5,5,S1_FT,TabTransformer,val,18.799693,0.341204,27.743028,0.317198,0.457742,0.011825
6,6,S2_IGBP_DOM,MLP,test,22.572415,0.261549,31.535028,0.399668,0.525652,0.011019
7,7,S2_IGBP_DOM,MLP,train,20.119199,0.029579,28.963975,0.190278,0.635289,0.000795
8,8,S2_IGBP_DOM,MLP,val,19.611607,0.120197,28.691840,0.326591,0.411350,0.008539
9,9,S2_IGBP_DOM,TabTransformer,test,22.144634,0.437319,30.877638,0.467663,0.547863,0.008886


# 5 合并 5 个模型的平均结果并导出

In [7]:
import pandas as pd
from pathlib import Path

out_dir = Path(r"d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs")

# 读 summary
ml_summary = pd.read_csv(out_dir / "lfmc_7schemes_ml_summary.csv")
dl_summary = pd.read_csv(out_dir / "lfmc_7schemes_dl_summary.csv")

# 读 raw（里面有 n）
ml_raw = pd.read_csv(out_dir / "lfmc_7schemes_ml_raw.csv")
dl_raw = pd.read_csv(out_dir / "lfmc_7schemes_dl_raw.csv")

# 提取每个 scheme/model/split 的样本数量
ml_counts = (
    ml_raw.groupby(["scheme", "model", "split"], as_index=False)["n"]
    .first()
    .rename(columns={"n": "n_samples"})
)

dl_counts = (
    dl_raw.groupby(["scheme", "model", "split"], as_index=False)["n"]
    .first()
    .rename(columns={"n": "n_samples"})
)

# merge 回 summary
ml_summary = ml_summary.merge(
    ml_counts, on=["scheme", "model", "split"], how="left"
)

dl_summary = dl_summary.merge(
    dl_counts, on=["scheme", "model", "split"], how="left"
)

# 合并 ML + DL
all_summary = pd.concat([ml_summary, dl_summary], axis=0, ignore_index=True)
all_summary = all_summary.sort_values(["scheme", "model", "split"]).reset_index(drop=True)

save_path = out_dir / "lfmc_7schemes_all_models_summary.csv"
all_summary.to_csv(save_path, index=False, encoding="utf-8-sig")

print("=== All Models Summary ===")
display(all_summary)

print(f"Saved to: {save_path}")


=== All Models Summary ===


,index,scheme,model,split,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R_mean,R_std,n_samples
0,0,S1_FT,CatBoost,test,19.178567,0.027228,28.095391,0.132122,0.634050,0.004386,3944
1,1,S1_FT,CatBoost,train,17.001684,0.195250,25.436676,0.239795,0.741157,0.005416,38435
2,2,S1_FT,CatBoost,val,16.843010,0.029992,25.487803,0.089818,0.554105,0.003780,4821
3,3,S1_FT,LightGBM,test,20.128022,0.011253,28.524253,0.097157,0.627783,0.002781,3944
4,4,S1_FT,LightGBM,train,11.532010,0.039900,19.708450,0.029744,0.857331,0.000443,38435
...,...,...,...,...,...,...,...,...,...,...,...
100,40,S7_LC_DOM_FE_W_CONS_NV,TabTransformer,train,21.160982,0.275904,30.089646,0.400400,0.604117,0.012708,26354
101,41,S7_LC_DOM_FE_W_CONS_NV,TabTransformer,val,22.007822,0.334158,30.625662,0.486150,0.661484,0.006592,2913
102,60,S7_LC_DOM_FE_W_CONS_NV,XGBoost,test,20.204767,0.145405,28.626162,0.138322,0.644609,0.004828,3085
103,61,S7_LC_DOM_FE_W_CONS_NV,XGBoost,train,9.614730,0.139776,17.578033,0.151158,0.891848,0.002318,26354


Saved to: d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs\lfmc_7schemes_all_models_summary.csv


# 6 基于CatBoost的S6方案，进行特征消融——Project312

In [1]:
!python ..\function\lfmc_ml\run_lfmc_ablation_batch.py --repeats 3


=== Running A0_FULL_S6 | exact S6 baseline: lc_dom + frac + entropy + weight + consistency ===
rows: 34667
split sizes: train=27802, val=3575, test=3290
weight min/max: 0.2000, 1.0000
num features: 29 (numeric=28, cat=1)
  [A0_FULL_S6] CatBoost repeat 1/3
  [A0_FULL_S6] CatBoost repeat 2/3
  [A0_FULL_S6] CatBoost repeat 3/3

=== Running A1_NO_CONS | remove consistency filter only ===
rows: 47200
split sizes: train=38435, val=4821, test=3944
weight min/max: 0.2000, 1.0000
num features: 29 (numeric=28, cat=1)
  [A1_NO_CONS] CatBoost repeat 1/3
  [A1_NO_CONS] CatBoost repeat 2/3
  [A1_NO_CONS] CatBoost repeat 3/3

=== Running A2_NO_WEIGHT | remove lc_weight only ===
rows: 34667
split sizes: train=27802, val=3575, test=3290
weight min/max: 1.0000, 1.0000
num features: 29 (numeric=28, cat=1)
  [A2_NO_WEIGHT] CatBoost repeat 1/3
  [A2_NO_WEIGHT] CatBoost repeat 2/3
  [A2_NO_WEIGHT] CatBoost repeat 3/3

=== Running A3_NO_FRAC_ENTROPY | remove lc_dom_frac and lc_entropy ===
rows: 34667
split 

In [2]:
import pandas as pd
from pathlib import Path

out_dir = Path(r"d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs_ablation")

# 读 summary
ab_summary = pd.read_csv(out_dir / "lfmc_s6_catboost_ablation_summary.csv")

# 读 raw（里面有 n）
ab_raw = pd.read_csv(out_dir / "lfmc_s6_catboost_ablation_raw.csv")

# 提取每个 ablation/model/split 的样本数量
ab_counts = (
    ab_raw.groupby(["ablation", "model", "split"], as_index=False)["n"]
    .first()
    .rename(columns={"n": "n_samples"})
)

# 如果 summary 里没有 ablation_desc，就从 raw 里补一份
if "ablation_desc" not in ab_summary.columns:
    ab_desc = (
        ab_raw.groupby(["ablation", "model"], as_index=False)["ablation_desc"]
        .first()
    )
    ab_summary = ab_summary.merge(
        ab_desc, on=["ablation", "model"], how="left"
    )

# merge 回 summary
ab_all_summary = ab_summary.merge(
    ab_counts, on=["ablation", "model", "split"], how="left"
)

# 排序
ab_all_summary = (
    ab_all_summary.sort_values(["ablation", "model", "split"])
    .reset_index(drop=True)
)

# 导出
save_path = out_dir / "lfmc_s6_catboost_ablation_all_summary.csv"
ab_all_summary.to_csv(save_path, index=False, encoding="utf-8-sig")

print("=== S6 CatBoost Ablation Summary ===")
display(ab_all_summary)

print(f"Saved to: {save_path}")


=== S6 CatBoost Ablation Summary ===


,index,ablation,model,split,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R_mean,R_std,ablation_desc,n_samples
0,0,A0_FULL_S6,CatBoost,test,18.005758,0.159541,25.435483,0.248833,0.737422,0.003219,exact S6 baseline: lc_dom + frac + entropy + w...,3290
1,1,A0_FULL_S6,CatBoost,train,16.944453,0.656461,25.374362,0.773322,0.753278,0.016788,exact S6 baseline: lc_dom + frac + entropy + w...,27802
2,2,A0_FULL_S6,CatBoost,val,20.054092,0.084172,28.147689,0.092944,0.663573,0.002813,exact S6 baseline: lc_dom + frac + entropy + w...,3575
3,3,A1_NO_CONS,CatBoost,test,19.104130,0.300810,27.774107,0.501476,0.652690,0.009203,remove consistency filter only,3944
4,4,A1_NO_CONS,CatBoost,train,18.506784,1.163672,27.235923,1.431902,0.700397,0.032568,remove consistency filter only,38435
5,5,A1_NO_CONS,CatBoost,val,17.559383,0.170977,26.078098,0.261353,0.523508,0.016838,remove consistency filter only,4821
6,6,A2_NO_WEIGHT,CatBoost,test,18.139302,0.058375,25.615377,0.035826,0.733109,0.002128,remove lc_weight only,3290
7,7,A2_NO_WEIGHT,CatBoost,train,17.359323,0.322643,25.904063,0.418743,0.741526,0.009033,remove lc_weight only,27802
8,8,A2_NO_WEIGHT,CatBoost,val,19.918244,0.076876,28.029354,0.083364,0.666781,0.003048,remove lc_weight only,3575
9,9,A3_NO_FRAC_ENTROPY,CatBoost,test,18.403151,0.216551,25.918625,0.316428,0.723982,0.006847,remove lc_dom_frac and lc_entropy,3290


Saved to: d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs_ablation\lfmc_s6_catboost_ablation_all_summary.csv


结论：
第一层，关键贡献项
* 季节/纬度变量（doy_sin, doy_cos, lat_norm）
* 一致性筛选（consistency filter）
* 核心生物物理变量（LAI / Hveg / SM / LST）

第二层，中等贡献项
* 熵（entropy）
* 样本权重（weight）
* IGBP_prop_ 数值特征*

第三层，边际贡献项
* dominant fraction（lc_dom_frac）
* VOD 特征


# 进一步的消融实验

In [3]:
!python ..\function\lfmc_ml\run_lfmc_ablation_v2_batch.py --repeats 3


=== Running FULL_S6 | exact S6 baseline: lc_dom + frac + entropy + weight + consistency ===
rows: 34667
split sizes: train=27802, val=3575, test=3290
weight min/max: 0.2000, 1.0000
num features: 29 (numeric=28, cat=1)
  [FULL_S6] CatBoost repeat 1/3
  [FULL_S6] CatBoost repeat 2/3
  [FULL_S6] CatBoost repeat 3/3

=== Running B1_NO_DOY_SIN | remove doy_sin only ===
rows: 34667
split sizes: train=27802, val=3575, test=3290
weight min/max: 0.2000, 1.0000
num features: 28 (numeric=27, cat=1)
  [B1_NO_DOY_SIN] CatBoost repeat 1/3
  [B1_NO_DOY_SIN] CatBoost repeat 2/3
  [B1_NO_DOY_SIN] CatBoost repeat 3/3

=== Running B2_NO_DOY_COS | remove doy_cos only ===
rows: 34667
split sizes: train=27802, val=3575, test=3290
weight min/max: 0.2000, 1.0000
num features: 28 (numeric=27, cat=1)
  [B2_NO_DOY_COS] CatBoost repeat 1/3
  [B2_NO_DOY_COS] CatBoost repeat 2/3
  [B2_NO_DOY_COS] CatBoost repeat 3/3

=== Running B3_NO_LAT_NORM | remove lat_norm only ===
rows: 34667
split sizes: train=27802, val=35

In [4]:
!python ..\function\lfmc_ml\run_lfmc_consistency_diagnose.py

Saved summary to: d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs_ablation\lfmc_s6_consistency_diagnose_summary.csv
Saved class details to: d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs_ablation\lfmc_s6_consistency_diagnose_by_class.csv
Saved FT details to: d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs_ablation\lfmc_s6_consistency_diagnose_by_ft.csv


In [5]:
import pandas as pd
from pathlib import Path

out_dir = Path(r"d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs_ablation")

# 读 summary
ab_summary = pd.read_csv(out_dir / "lfmc_s6_catboost_ablation_v2_summary.csv")

# 读 raw（里面有 n）
ab_raw = pd.read_csv(out_dir / "lfmc_s6_catboost_ablation_v2_raw.csv")

# 提取每个 ablation/model/split 的样本数量
ab_counts = (
    ab_raw.groupby(["ablation", "model", "split"], as_index=False)["n"]
    .first()
    .rename(columns={"n": "n_samples"})
)

# 如果 summary 里没有 ablation_desc，就从 raw 里补一份
if "ablation_desc" not in ab_summary.columns:
    ab_desc = (
        ab_raw.groupby(["ablation", "model"], as_index=False)["ablation_desc"]
        .first()
    )
    ab_summary = ab_summary.merge(
        ab_desc, on=["ablation", "model"], how="left"
    )

# merge 回 summary
ab_all_summary = ab_summary.merge(
    ab_counts, on=["ablation", "model", "split"], how="left"
)

# 排序
ab_all_summary = (
    ab_all_summary.sort_values(["ablation", "model", "split"])
    .reset_index(drop=True)
)

# 导出
save_path = out_dir / "lfmc_s6_catboost_ablation_v2_all_summary.csv"
ab_all_summary.to_csv(save_path, index=False, encoding="utf-8-sig")

print("=== S6 CatBoost Ablation V2 Summary ===")
display(ab_all_summary)

print(f"Saved to: {save_path}")


=== S6 CatBoost Ablation V2 Summary ===


,index,ablation,model,split,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R_mean,R_std,ablation_desc,n_samples
0,0,B1_NO_DOY_SIN,CatBoost,test,20.130549,0.164799,28.706800,0.152984,0.651102,0.006058,remove doy_sin only,3290
1,1,B1_NO_DOY_SIN,CatBoost,train,18.244977,0.242116,27.575455,0.285426,0.704873,0.007405,remove doy_sin only,27802
2,2,B1_NO_DOY_SIN,CatBoost,val,22.218549,0.067253,31.164557,0.036916,0.571903,0.002510,remove doy_sin only,3575
3,3,B2_NO_DOY_COS,CatBoost,test,18.425525,0.081407,25.830756,0.049367,0.727143,0.001773,remove doy_cos only,3290
4,4,B2_NO_DOY_COS,CatBoost,train,17.079674,0.260525,25.557604,0.331757,0.749449,0.007549,remove doy_cos only,27802
5,5,B2_NO_DOY_COS,CatBoost,val,20.195462,0.061478,28.299315,0.056947,0.658316,0.002046,remove doy_cos only,3575
6,6,B3_NO_LAT_NORM,CatBoost,test,20.073277,0.031314,28.034005,0.117758,0.670141,0.002302,remove lat_norm only,3290
7,7,B3_NO_LAT_NORM,CatBoost,train,17.807144,0.606908,26.520591,0.721623,0.726908,0.018032,remove lat_norm only,27802
8,8,B3_NO_LAT_NORM,CatBoost,val,21.256606,0.049321,29.422413,0.032985,0.624115,0.002198,remove lat_norm only,3575
9,9,B4_NO_LAI,CatBoost,test,18.366481,0.008348,26.004178,0.051551,0.723401,0.003054,remove LAI only,3290


Saved to: d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs_ablation\lfmc_s6_catboost_ablation_v2_all_summary.csv


## S6 + CatBoost 细粒度消融结果

以 `FULL_S6` 为基线，细粒度消融结果表明，季节项仍是最重要的信息来源。其中，去除 `doy_sin` 后性能下降最明显，Validation RMSE 从 `28.15` 上升到 `31.16`，Test RMSE 从 `25.44` 上升到 `28.71`，说明 `doy_sin` 是最关键的单变量。相比之下，去除 `doy_cos` 仅带来较小性能损失，而去除 `lat_norm` 也会显著降低精度，说明纬度背景同样对 LFMC 预测具有重要作用。总体来看，季节项内部的重要性大致为：`doy_sin > lat_norm > doy_cos`。

在核心生物物理变量中，`LAI` 和 `Hveg` 的贡献明显高于 `LST` 和 `SM`。去除 `LAI` 或 `Hveg` 均会导致验证集和测试集误差上升，而去除 `SM` 后性能几乎没有变化，说明 `SM` 在当前 S6 + CatBoost 框架中的边际贡献较弱。整体上，核心变量的重要性可概括为：`LAI ≈ Hveg > LST >> SM`。

此外，`NO_CONS` 结果再次表明，去掉 consistency 约束后验证集误差下降，但测试集误差显著上升，说明 consistency 过滤更像是一种提高泛化稳健性的样本质量控制机制，而不是单纯优化验证集分数的手段。综合来看，当前模型最依赖的关键信息是 `doy_sin`、`lat_norm`、`LAI`、`Hveg` 以及 consistency 约束。


# 根据最终消融结果选择一套方案，再次尝试所有模型

# 机器学习部分

In [6]:
!python ..\function\lfmc_ml\run_lfmc_final_ml_batch.py --repeats 3

=== Final S6-lite dataset ===
rows: 34667
train=27802, val=3575, test=3290
num features: 13 (numeric=12, cat=1)
numeric columns: ['VOD_Ku_Hpol_Asc', 'VOD_Ku_Vpol_Asc', 'VOD_X_Hpol_Asc', 'VOD_X_Vpol_Asc', 'VOD_C_Hpol_Asc', 'VOD_C_Vpol_Asc', 'LAI', 'Hveg', 'LST', 'doy_sin', 'doy_cos', 'lat_norm']
categorical column: lc_dom
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001059 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2975
[LightGBM] [Info] Number of data points in the train set: 27802, number of used features: 13
[LightGBM] [Info] Start training from score 106.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

# 深度学习部分 

In [3]:
# 第一部分代码
!python ..\function\lfmc_ml\run_lfmc_final_dl_v2_batch.py --repeats 3

Device: cuda
=== Final S6-lite DL v2 dataset ===
rows: 34667
train=27802, val=3575, test=3290
num features: 13 (numeric=12, cat=1)
numeric columns: ['VOD_Ku_Hpol_Asc', 'VOD_Ku_Vpol_Asc', 'VOD_X_Hpol_Asc', 'VOD_X_Vpol_Asc', 'VOD_C_Hpol_Asc', 'VOD_C_Vpol_Asc', 'LAI', 'Hveg', 'LST', 'doy_sin', 'doy_cos', 'lat_norm']
categorical column: lc_dom
n_cats: 10
[repeat 1/3] training MLP_base
[repeat 1/3] training MLP_deep
[repeat 1/3] training ResNetMLP
[repeat 1/3] training TabTransformer_base
[repeat 1/3] training FTTransformer
[repeat 1/3] training TabNet
[repeat 2/3] training MLP_base
[repeat 2/3] training MLP_deep
[repeat 2/3] training ResNetMLP
[repeat 2/3] training TabTransformer_base
[repeat 2/3] training FTTransformer
[repeat 2/3] training TabNet
[repeat 3/3] training MLP_base
[repeat 3/3] training MLP_deep
[repeat 3/3] training ResNetMLP
[repeat 3/3] training TabTransformer_base
[repeat 3/3] training FTTransformer
[repeat 3/3] training TabNet
Saved raw results to: d:\Python\jupyter\jupy

c:\Users\Ronin\.conda\envs\tau_omega_env\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
c:\Users\Ronin\.conda\envs\tau_omega_env\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
c:\Users\Ronin\.conda\envs\tau_omega_env\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [2]:
# 第二部分代码，尝试Mixture of Experts
!python ..\function\lfmc_ml\run_lfmc_final_dl_moe_batch.py --repeats 3

Device: cuda
=== Final S6-lite DL MoE dataset ===
rows: 34667
train=27802, val=3575, test=3290
num features: 13 (numeric=12, cat=1)
numeric columns: ['VOD_Ku_Hpol_Asc', 'VOD_Ku_Vpol_Asc', 'VOD_X_Hpol_Asc', 'VOD_X_Vpol_Asc', 'VOD_C_Hpol_Asc', 'VOD_C_Vpol_Asc', 'LAI', 'Hveg', 'LST', 'doy_sin', 'doy_cos', 'lat_norm']
categorical column: lc_dom
n_cats: 10
[repeat 1/3] training MoE_MLP
[repeat 1/3] training MoE_ResNetMLP
[repeat 2/3] training MoE_MLP
[repeat 2/3] training MoE_ResNetMLP
[repeat 3/3] training MoE_MLP
[repeat 3/3] training MoE_ResNetMLP
Saved raw results to: d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs_final\lfmc_final_s6_lite_dl_moe_raw.csv
Saved summary results to: d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs_final\lfmc_final_s6_lite_dl_moe_summary.csv


In [5]:
import pandas as pd
from pathlib import Path

out_dir = Path(r"d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs_final")

# 读 summary
ml_summary = pd.read_csv(out_dir / "lfmc_final_s6_lite_ml_summary.csv")
dl_v2_summary = pd.read_csv(out_dir / "lfmc_final_s6_lite_dl_v2_summary.csv")
moe_summary = pd.read_csv(out_dir / "lfmc_final_s6_lite_dl_moe_summary.csv")

# 读 raw（里面有 n）
ml_raw = pd.read_csv(out_dir / "lfmc_final_s6_lite_ml_raw.csv")
dl_v2_raw = pd.read_csv(out_dir / "lfmc_final_s6_lite_dl_v2_raw.csv")
moe_raw = pd.read_csv(out_dir / "lfmc_final_s6_lite_dl_moe_raw.csv")

# 提取每个 scheme/model/split 的样本数量
ml_counts = (
    ml_raw.groupby(["scheme", "model", "split"], as_index=False)["n"]
    .first()
    .rename(columns={"n": "n_samples"})
)

dl_v2_counts = (
    dl_v2_raw.groupby(["scheme", "model", "split"], as_index=False)["n"]
    .first()
    .rename(columns={"n": "n_samples"})
)

moe_counts = (
    moe_raw.groupby(["scheme", "model", "split"], as_index=False)["n"]
    .first()
    .rename(columns={"n": "n_samples"})
)

# merge 回 summary
ml_summary = ml_summary.merge(
    ml_counts, on=["scheme", "model", "split"], how="left"
)

dl_v2_summary = dl_v2_summary.merge(
    dl_v2_counts, on=["scheme", "model", "split"], how="left"
)

moe_summary = moe_summary.merge(
    moe_counts, on=["scheme", "model", "split"], how="left"
)

# 合并所有结果
all_summary = pd.concat(
    [ml_summary, dl_v2_summary, moe_summary],
    axis=0,
    ignore_index=True
)

all_summary = all_summary.sort_values(["scheme", "model", "split"]).reset_index(drop=True)

# 导出
save_path = out_dir / "lfmc_final_s6_lite_all_models_summary.csv"
all_summary.to_csv(save_path, index=False, encoding="utf-8-sig")

print("=== Final S6-lite All Models Summary ===")
display(all_summary)

print(f"Saved to: {save_path}")


=== Final S6-lite All Models Summary ===


,index,scheme,model,split,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R_mean,R_std,n_samples
0,0,FINAL_S6_LITE,CatBoost,test,18.190571,0.087100,25.608331,0.062102,0.731371,0.002479,3290
1,1,FINAL_S6_LITE,CatBoost,train,17.571309,0.134397,26.151114,0.156208,0.732886,0.003896,27802
2,2,FINAL_S6_LITE,CatBoost,val,19.594323,0.092091,27.736873,0.072065,0.674000,0.002319,3575
3,0,FINAL_S6_LITE,FTTransformer,test,18.946645,0.325304,26.702084,0.463090,0.684966,0.023680,3290
4,1,FINAL_S6_LITE,FTTransformer,train,18.905624,0.280680,27.751168,0.467969,0.681167,0.013070,27802
5,2,FINAL_S6_LITE,FTTransformer,val,20.663797,0.369331,29.272924,0.538765,0.625435,0.018312,3575
6,3,FINAL_S6_LITE,LightGBM,test,18.593557,0.103270,26.190473,0.142493,0.708935,0.004908,3290
7,4,FINAL_S6_LITE,LightGBM,train,12.026476,0.056872,20.592860,0.090816,0.849907,0.001584,27802
8,5,FINAL_S6_LITE,LightGBM,val,20.637382,0.116313,29.032938,0.270392,0.630742,0.007677,3575
9,3,FINAL_S6_LITE,MLP_base,test,19.481013,0.196609,26.938167,0.276936,0.679697,0.003866,3290


Saved to: d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs_final\lfmc_final_s6_lite_all_models_summary.csv


### 最终结果总结

最终结果已经很明确：在 **S6-lite** 方案下，**`TabNet` 成为当前最优模型**，其 test 集表现优于 `CatBoost`，说明在合适的数据定义与特征裁剪后，深度学习模型已经能够超过最优树模型。

具体来看，`TabNet` 在验证集上与 `CatBoost` 非常接近，但在测试集上进一步拉开差距，因此可视为当前最优的最终生产模型。与此同时，**`CatBoost` 仍然是最稳健的传统基线**，具有较强的稳定性和可解释性；**`MoE_MLP`** 也取得了接近 `CatBoost` 的结果，表明专家模型方向具有真实潜力。

相比之下，`FTTransformer`、`TabTransformer_base`、`ResNetMLP` 等更复杂结构并未展现出优势，说明在当前任务设定下，Transformer/残差路线并不是最优选择。

**因此，当前推荐的最终方案为：`S6-lite + TabNet`；同时保留 `S6-lite + CatBoost` 作为稳健传统对照，保留 `S6-lite + MoE_MLP` 作为研究型神经网络对照。**

# 虚拟环境：lfmc_infer

In [13]:
# 训练 ML：
!python ..\function\lfmc_ml\run_lfmc_ml_prod.py --repeats 1

Running: python D:\Python\jupyter\jupyter\LFMCRegressor\function\lfmc_ml\run_lfmc_final_ml_batch.py --repeats 1
=== Final S6-lite dataset ===
rows: 34667
train=27802, val=3575, test=3290
num features: 13 (numeric=12, cat=1)
numeric columns: ['VOD_Ku_Hpol_Asc', 'VOD_Ku_Vpol_Asc', 'VOD_X_Hpol_Asc', 'VOD_X_Vpol_Asc', 'VOD_C_Hpol_Asc', 'VOD_C_Vpol_Asc', 'LAI', 'Hveg', 'LST', 'doy_sin', 'doy_cos', 'lat_norm']
categorical column: lc_dom
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001363 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2975
[LightGBM] [Info] Number of data points in the train set: 27802, number of used features: 13
[LightGBM] [Info] Start training from score 106.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightG

In [15]:
# 训练 DL：
!python ..\function\lfmc_ml\run_lfmc_dl_prod.py --repeats 1

Device: cuda
=== Final S6-lite DL product dataset ===
rows: 34667
train=27802, val=3575, test=3290
num features: 13 (numeric=12, cat=1)
numeric columns: ['VOD_Ku_Hpol_Asc', 'VOD_Ku_Vpol_Asc', 'VOD_X_Hpol_Asc', 'VOD_X_Vpol_Asc', 'VOD_C_Hpol_Asc', 'VOD_C_Vpol_Asc', 'LAI', 'Hveg', 'LST', 'doy_sin', 'doy_cos', 'lat_norm']
categorical column: lc_dom
[repeat 1/1] training ResNetMLP
[repeat 1/1] training TabNet
[repeat 1/1] training MoE_ResNetMLP
Saved raw results to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl\lfmc_final_s6_lite_dl_raw.csv
Saved summary results to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl\lfmc_final_s6_lite_dl_summary.csv


In [18]:
# ml预测（单日测试）
!python D:\Python\jupyter\jupyter\LFMCRegressor\function\lfmc_ml\run_lfmc_ml_prod.py --mode predict --input-mode raster --date 20200115 --model-name CatBoost --artifact-dir "D:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\ml\CatBoost_rep1" --output-path "G:\data\LFMC\data-Catboost\lfmc_pred_20200115.h5" --nonveg-threshold 0.7

[OK] Raster prediction finished -> G:\data\LFMC\data-Catboost\lfmc_pred_20200115.h5


2026-04-07 12:56:19,838 - INFO - 开始构造 2020-01-15 的产品输入表，共 6,480,000 个像元
2026-04-07 12:56:19,949 - INFO - Hveg 加载成功: Hveg, shape=(1800, 3600)
2026-04-07 12:56:21,686 - INFO - 8日 LAI 可用日期数: 1012
2026-04-07 12:56:25,887 - INFO - LC 加载成功: 2020


In [ ]:
# ml预测（批量日期）
from pathlib import Path
import pandas as pd

script = r"D:\Python\jupyter\jupyter\LFMCRegressor\function\lfmc_ml\run_lfmc_ml_prod.py"
artifact_dir = r"D:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\ml\CatBoost_rep1"
out_dir = Path(r"G:\data\LFMC\data-Catboost")
out_dir.mkdir(parents=True, exist_ok=True)

dates = pd.date_range("2020-01-01", "2020-12-31", freq="D")

for dt in dates:
    date_str = dt.strftime("%Y%m%d")
    out_path = out_dir / f"lfmc_pred_{date_str}.h5"

    !python {script} --mode predict --input-mode raster --date {date_str} --model-name CatBoost --artifact-dir "{artifact_dir}" --output-path "{out_path}" --nonveg-threshold 0.7


In [ ]:
# DL 预测（TabNet）：
!python D:\Python\jupyter\jupyter\LFMCRegressor\function\lfmc_ml\run_lfmc_dl_prod.py --mode predict --input-mode raster --date 20200115 --model-name TabNet --artifact-dir "D:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl\TabNet_rep1" --output-path "G:\data\LFMC\data-TabNet\lfmc_pred_20200115.h5" --nonveg-threshold 0.7

[OK] Raster prediction finished -> G:\data\LFMC\data-TabNet\lfmc_pred_20200115.h5


2026-04-07 13:03:40,237 - INFO - 开始构造 2020-01-15 的产品输入表，共 6,480,000 个像元
2026-04-07 13:03:40,372 - INFO - Hveg 加载成功: Hveg, shape=(1800, 3600)
2026-04-07 13:03:42,077 - INFO - 8日 LAI 可用日期数: 1012
2026-04-07 13:03:46,488 - INFO - LC 加载成功: 2020


In [21]:
# DL 预测（ResNetMLP）：
!python D:\Python\jupyter\jupyter\LFMCRegressor\function\lfmc_ml\run_lfmc_dl_prod.py --mode predict --input-mode raster --date 20200115 --model-name ResNetMLP --artifact-dir "D:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl\ResNetMLP_rep1" --output-path "G:\data\LFMC\data-ResNetMLP\lfmc_pred_20200115.h5" --nonveg-threshold 0.7

[OK] Raster prediction finished -> G:\data\LFMC\data-ResNetMLP\lfmc_pred_20200115.h5


2026-04-07 13:33:47,634 - INFO - 开始构造 2020-01-15 的产品输入表，共 6,480,000 个像元
2026-04-07 13:33:47,752 - INFO - Hveg 加载成功: Hveg, shape=(1800, 3600)
2026-04-07 13:33:49,461 - INFO - 8日 LAI 可用日期数: 1012
2026-04-07 13:33:53,862 - INFO - LC 加载成功: 2020


In [22]:
# DL 预测（MoE_ResNetMLP）：
!python D:\Python\jupyter\jupyter\LFMCRegressor\function\lfmc_ml\run_lfmc_dl_prod.py --mode predict --input-mode raster --date 20200115 --model-name MoE_ResNetMLP --artifact-dir "D:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl\MoE_ResNetMLP_rep1" --output-path "G:\data\LFMC\data-MoE_ResNetMLP\lfmc_pred_20200115.h5" --nonveg-threshold 0.7

[OK] Raster prediction finished -> G:\data\LFMC\data-MoE_ResNetMLP\lfmc_pred_20200115.h5


2026-04-07 13:34:26,207 - INFO - 开始构造 2020-01-15 的产品输入表，共 6,480,000 个像元
2026-04-07 13:34:26,331 - INFO - Hveg 加载成功: Hveg, shape=(1800, 3600)
2026-04-07 13:34:28,116 - INFO - 8日 LAI 可用日期数: 1012
2026-04-07 13:34:32,494 - INFO - LC 加载成功: 2020


# 绘制一份地图，展示目前数据的划分

In [23]:
!python ..\function\lfmc_ml\plot_s6_lite_split_map.py

Saved block table to: d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs_final\split_maps\s6_lite_split_blocks.csv
Saved map to: d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs_final\split_maps\s6_lite_split_map_all.png
Saved train/val map to: d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs_final\split_maps\s6_lite_split_map_train_val.png
Saved summary to: d:\Python\jupyter\jupyter\LFMCRegressor\figs\batch_runs_final\split_maps\s6_lite_split_block_summary.csv


由于某个样本所在站点没有足够长的历史序列，它就不会进入 Temporal CNN 训练集。所以我们检查样本分布

In [34]:
!python ..\function\lfmc_ml\plot_temporal_cnn_sample_map.py --window-size 4


Saved block table to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl_temporal\temporal_cnn_sample_blocks.csv
Saved map to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl_temporal\temporal_cnn_sample_map_all.png
Saved train/val map to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl_temporal\temporal_cnn_sample_map_train_val.png
Saved sample summary to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl_temporal\temporal_cnn_sample_summary.csv
Saved block summary to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl_temporal\temporal_cnn_block_summary.csv


# 现在4个模型（Catboost、TabNet、ResNet、MoE-ResNet）的绘图，经过观察，CatBoost是遵循VOD观测为缺失值的情况进行绘制的，其他3个模型设置VOD缺失值为0，从而有每日大部分区域的结果（将VOD当做辅助的信息）。
目前只有TabNet的结果最接近全球的真实情况，CatBoost一般，而两个ResNet在中高纬度会出现明显地纬度分带情况

后续的处理建议：
1.去掉纬度和doy？
2.如果一个像元有多个物种，而不同物种 LFMC 差异大，就剔除；只有多个物种时间变化相近（任意两物种 Pearson r ≥ 0.5）才保留（举个例子（Xie等人的文章）），不过这个是根据逐站点来实验的，也就是一个站点有多次测量。
3.输入包含静态信息，分析深度学习如何同时处理动态和静态的信息
4.加入更多的光学信号
5.全球可拓展性的尝试——分类型建立模型（MoE）。


# 尝试静态信息的分流处理：双分支Temporal CNN

In [29]:
!python ..\function\lfmc_ml\run_lfmc_final_temporal_cnn_batch.py --repeats 1 --window-size 4


Device: cuda
=== Final S6-lite Dual-Branch Temporal CNN dataset ===
window_size: 4
rows: 32051
train=25707, val=3317, test=3027
static numeric columns: ['Hveg', 'lat_norm']
dynamic columns: ['VOD_Ku_Hpol_Asc', 'VOD_Ku_Vpol_Asc', 'VOD_X_Hpol_Asc', 'VOD_X_Vpol_Asc', 'VOD_C_Hpol_Asc', 'VOD_C_Vpol_Asc', 'LAI', 'LST', 'doy_sin', 'doy_cos']
categorical column: lc_dom
[repeat 1/1] training DualBranchTemporalCNN
[repeat 1] DualBranchTemporalCNN metrics
  train MAE=19.3275  RMSE=27.7572  R=0.6738
  val   MAE=20.6166  RMSE=28.7841  R=0.6367
  test  MAE=19.6089  RMSE=27.1280  R=0.6491
Saved raw results to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl_temporal\lfmc_final_s6_lite_temporal_cnn_raw.csv
Saved summary results to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl_temporal\lfmc_final_s6_lite_temporal_cnn_summary.csv


In [ ]:
!python ..\function\lfmc_ml\run_lfmc_temporal_cnn_prod.py --artifact-dir "..\notebooks\artifacts\dl_temporal\DualBranchTemporalCNN_rep1" --date 20200115 --output-path "G:\data\LFMC\data-TemporalCNN\lfmc_pred_20200115.h5" --nonveg-threshold 0.7

2026-04-08 18:03:35,758 - INFO - 开始构造 2020-01-12 的产品输入表，共 6,480,000 个像元
2026-04-08 18:03:35,916 - INFO - Hveg 加载成功: Hveg, shape=(1800, 3600)
2026-04-08 18:03:37,701 - INFO - 8日 LAI 可用日期数: 1012
2026-04-08 18:03:42,177 - INFO - LC 加载成功: 2020
2026-04-08 18:04:01,511 - INFO - 开始构造 2020-01-13 的产品输入表，共 6,480,000 个像元
2026-04-08 18:04:27,811 - INFO - 开始构造 2020-01-14 的产品输入表，共 6,480,000 个像元
2026-04-08 18:04:54,089 - INFO - 开始构造 2020-01-15 的产品输入表，共 6,480,000 个像元


[OK] Temporal raster prediction finished -> G:\data\LFMC\data-TemporalCNN\lfmc_pred_20200115.h5


# 实验建议：样本方面
现在我们的样本是按照严格地空间独立性进行划分的


学习到一种样本划分的思路（Zhu，2022）：
X1-X2年：训练 X2-X3年：验证。训练集和验证集存在空间重合，不过存在部分点在训练集中是没有出现过的


受此启发，使用一个新的实验方案进行对照：
全球的LFMC field data，存在极度地样本分布不均的情况，以美国和欧洲的数据居多，分布较广（如图所示），我们在训练集只选取2002-2018年的美国的站点，然后验证集包含2019-2023年美国的站点，然后剩余大洲的所有点作为测试集（外部）。这样如果实验成功，可以让模型具有更强的全球可迁移性

3个精度：
1.美国区域训练集精度
2.美国区域验证集（与训练集空间重合）的精度
3.美国区域验证集（不与训练集空间重合）的精度
4.其他大洲测试集的精度


# 实验建议：模型架构方面
1.先做时序编码，用Temporal CNN提取动态时序特征，再交给之前的CatBoost。
2.修改目前Temporal CNN的静态分支为我们之前使用的内容。
3.将Temporal CNN进行MoE操作。

# 实验设置
（先全部基于新split）

新 split + 现有模型
新 split + 现有模型 + MoE
新 split + Temporal CNN
新 split + Temporal CNN + MoE
新 split + Temporal CNN + ResNetStatic
新 split + Temporal CNN + ResNetStatic + MoE
新 split + Temporal CNN + TabNetStatic
新 split + Temporal CNN + TabNetStatic + MoE

In [43]:
# 1. 新 split + 现有模型
# 这里一次运行 2 个模型：ResNetMLP, TabNet
!python ..\function\lfmc_ml\run_lfmc_us_transfer_dl_batch.py --repeats 1 --models "ResNetMLP,TabNet"


Device: cuda
=== US Transfer S6-lite dataset ===
rows: 34667
numeric columns: ['VOD_Ku_Hpol_Asc', 'VOD_Ku_Vpol_Asc', 'VOD_X_Hpol_Asc', 'VOD_X_Vpol_Asc', 'VOD_C_Hpol_Asc', 'VOD_C_Vpol_Asc', 'LAI', 'Hveg', 'LST', 'doy_sin', 'doy_cos', 'lat_norm']
categorical column: lc_dom
train_us: n=23519, blocks=460
val_us_overlap: n=9723, blocks=379
val_us_nonoverlap: n=661, blocks=63
test_external: n=764, blocks=82
[repeat 1/1] training ResNetMLP
[repeat 1] ResNetMLP metrics
  train_us           MAE=17.7669  RMSE=25.7765  R=0.7259
  val_us_overlap     MAE=18.3279  RMSE=26.1720  R=0.6801
  val_us_nonoverlap  MAE=31.8261  RMSE=42.6817  R=0.2134
  test_external      MAE=38.5809  RMSE=51.2905  R=0.3400
[repeat 1/1] training TabNet
[repeat 1] TabNet metrics
  train_us           MAE=15.3217  RMSE=23.4200  R=0.7919
  val_us_overlap     MAE=17.5485  RMSE=25.9134  R=0.6902
  val_us_nonoverlap  MAE=26.2010  RMSE=37.5645  R=0.3658
  test_external      MAE=140.8493  RMSE=220.0858  R=0.0095
Saved raw results to:

In [44]:
# 2. 新 split + 现有模型 + MoE
# 这里运行 1 个模型：MoE_ResNetMLP
!python ..\function\lfmc_ml\run_lfmc_us_transfer_dl_batch.py --repeats 1 --models "MoE_ResNetMLP"


Device: cuda
=== US Transfer S6-lite dataset ===
rows: 34667
numeric columns: ['VOD_Ku_Hpol_Asc', 'VOD_Ku_Vpol_Asc', 'VOD_X_Hpol_Asc', 'VOD_X_Vpol_Asc', 'VOD_C_Hpol_Asc', 'VOD_C_Vpol_Asc', 'LAI', 'Hveg', 'LST', 'doy_sin', 'doy_cos', 'lat_norm']
categorical column: lc_dom
train_us: n=23519, blocks=460
val_us_overlap: n=9723, blocks=379
val_us_nonoverlap: n=661, blocks=63
test_external: n=764, blocks=82
[repeat 1/1] training MoE_ResNetMLP
[repeat 1] MoE_ResNetMLP metrics
  train_us           MAE=18.0469  RMSE=25.9138  R=0.7228
  val_us_overlap     MAE=18.4100  RMSE=26.0814  R=0.6843
  val_us_nonoverlap  MAE=27.5738  RMSE=35.7486  R=0.4362
  test_external      MAE=41.4120  RMSE=52.6441  R=0.3379
Saved raw results to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl_us_transfer\lfmc_us_transfer_dl_raw.csv
Saved summary results to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl_us_transfer\lfmc_us_transfer_dl_summary.csv


In [45]:
# 3. 新 split + Temporal CNN
# 这里运行 1 个模型：TemporalCNN_MLPStatic
!python ..\function\lfmc_ml\run_lfmc_us_transfer_temporal_batch.py --repeats 1 --window-size 4 --models "TemporalCNN_MLPStatic"


Device: cuda
=== US Transfer S6-lite dataset ===
rows: 34667
numeric columns: ['VOD_Ku_Hpol_Asc', 'VOD_Ku_Vpol_Asc', 'VOD_X_Hpol_Asc', 'VOD_X_Vpol_Asc', 'VOD_C_Hpol_Asc', 'VOD_C_Vpol_Asc', 'LAI', 'Hveg', 'LST', 'doy_sin', 'doy_cos', 'lat_norm']
categorical column: lc_dom
train_us: n=23519, blocks=460
val_us_overlap: n=9723, blocks=379
val_us_nonoverlap: n=661, blocks=63
test_external: n=764, blocks=82
=== US Transfer Temporal dataset ===
window_size: 4
rows: 32051
train_us: n=21578, sites=601
val_us_overlap: n=9530, sites=529
val_us_nonoverlap: n=466, sites=50
test_external: n=477, sites=36
[repeat 1/1] training TemporalCNN_MLPStatic
[repeat 1] TemporalCNN_MLPStatic metrics
  train_us           MAE=18.9195  RMSE=27.0660  R=0.7044
  val_us_overlap     MAE=18.5921  RMSE=26.5335  R=0.6672
  val_us_nonoverlap  MAE=24.0305  RMSE=35.8607  R=0.4196
  test_external      MAE=288.5285  RMSE=533.7790  R=0.1530
Saved raw results to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl_us

In [46]:
# 4. 新 split + Temporal CNN + MoE
# 这里运行 1 个模型：TemporalCNN_MoE
!python ..\function\lfmc_ml\run_lfmc_us_transfer_temporal_batch.py --repeats 1 --window-size 4 --models "TemporalCNN_MoE"


Device: cuda
=== US Transfer S6-lite dataset ===
rows: 34667
numeric columns: ['VOD_Ku_Hpol_Asc', 'VOD_Ku_Vpol_Asc', 'VOD_X_Hpol_Asc', 'VOD_X_Vpol_Asc', 'VOD_C_Hpol_Asc', 'VOD_C_Vpol_Asc', 'LAI', 'Hveg', 'LST', 'doy_sin', 'doy_cos', 'lat_norm']
categorical column: lc_dom
train_us: n=23519, blocks=460
val_us_overlap: n=9723, blocks=379
val_us_nonoverlap: n=661, blocks=63
test_external: n=764, blocks=82
=== US Transfer Temporal dataset ===
window_size: 4
rows: 32051
train_us: n=21578, sites=601
val_us_overlap: n=9530, sites=529
val_us_nonoverlap: n=466, sites=50
test_external: n=477, sites=36
[repeat 1/1] training TemporalCNN_MoE
[repeat 1] TemporalCNN_MoE metrics
  train_us           MAE=19.4211  RMSE=27.6784  R=0.6964
  val_us_overlap     MAE=19.1035  RMSE=27.1672  R=0.6543
  val_us_nonoverlap  MAE=25.0176  RMSE=35.6683  R=0.4101
  test_external      MAE=352.7869  RMSE=656.6936  R=0.1557
Saved raw results to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl_us_transfer_te

In [47]:
# 5. 新 split + Temporal CNN + ResNetStatic
# 这里运行 1 个模型：TemporalCNN_ResNetStatic
!python ..\function\lfmc_ml\run_lfmc_us_transfer_temporal_batch.py --repeats 1 --window-size 4 --models "TemporalCNN_ResNetStatic"


Device: cuda
=== US Transfer S6-lite dataset ===
rows: 34667
numeric columns: ['VOD_Ku_Hpol_Asc', 'VOD_Ku_Vpol_Asc', 'VOD_X_Hpol_Asc', 'VOD_X_Vpol_Asc', 'VOD_C_Hpol_Asc', 'VOD_C_Vpol_Asc', 'LAI', 'Hveg', 'LST', 'doy_sin', 'doy_cos', 'lat_norm']
categorical column: lc_dom
train_us: n=23519, blocks=460
val_us_overlap: n=9723, blocks=379
val_us_nonoverlap: n=661, blocks=63
test_external: n=764, blocks=82
=== US Transfer Temporal dataset ===
window_size: 4
rows: 32051
train_us: n=21578, sites=601
val_us_overlap: n=9530, sites=529
val_us_nonoverlap: n=466, sites=50
test_external: n=477, sites=36
[repeat 1/1] training TemporalCNN_ResNetStatic
[repeat 1] TemporalCNN_ResNetStatic metrics
  train_us           MAE=17.7656  RMSE=25.6094  R=0.7360
  val_us_overlap     MAE=17.8530  RMSE=25.6346  R=0.6918
  val_us_nonoverlap  MAE=23.5330  RMSE=35.7566  R=0.4030
  test_external      MAE=32.9538  RMSE=47.8362  R=0.2063
Saved raw results to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\d

In [48]:
# 6. 新 split + Temporal CNN + ResNetStatic + MoE
# 这里运行 1 个模型：TemporalCNN_ResNetStatic_MoE
!python ..\function\lfmc_ml\run_lfmc_us_transfer_temporal_batch.py --repeats 1 --window-size 4 --models "TemporalCNN_ResNetStatic_MoE"


Device: cuda
=== US Transfer S6-lite dataset ===
rows: 34667
numeric columns: ['VOD_Ku_Hpol_Asc', 'VOD_Ku_Vpol_Asc', 'VOD_X_Hpol_Asc', 'VOD_X_Vpol_Asc', 'VOD_C_Hpol_Asc', 'VOD_C_Vpol_Asc', 'LAI', 'Hveg', 'LST', 'doy_sin', 'doy_cos', 'lat_norm']
categorical column: lc_dom
train_us: n=23519, blocks=460
val_us_overlap: n=9723, blocks=379
val_us_nonoverlap: n=661, blocks=63
test_external: n=764, blocks=82
=== US Transfer Temporal dataset ===
window_size: 4
rows: 32051
train_us: n=21578, sites=601
val_us_overlap: n=9530, sites=529
val_us_nonoverlap: n=466, sites=50
test_external: n=477, sites=36
[repeat 1/1] training TemporalCNN_ResNetStatic_MoE
[repeat 1] TemporalCNN_ResNetStatic_MoE metrics
  train_us           MAE=17.1658  RMSE=25.1546  R=0.7506
  val_us_overlap     MAE=17.5915  RMSE=25.6831  R=0.6938
  val_us_nonoverlap  MAE=24.0274  RMSE=36.8457  R=0.3546
  test_external      MAE=34.9245  RMSE=51.3557  R=0.1780
Saved raw results to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\art

In [49]:
# 7. 新 split + Temporal CNN + TabNetStatic
# 这里运行 1 个模型：TemporalCNN_TabNetStatic
!python ..\function\lfmc_ml\run_lfmc_us_transfer_temporal_batch.py --repeats 1 --window-size 4 --models "TemporalCNN_TabNetStatic"


Device: cuda
=== US Transfer S6-lite dataset ===
rows: 34667
numeric columns: ['VOD_Ku_Hpol_Asc', 'VOD_Ku_Vpol_Asc', 'VOD_X_Hpol_Asc', 'VOD_X_Vpol_Asc', 'VOD_C_Hpol_Asc', 'VOD_C_Vpol_Asc', 'LAI', 'Hveg', 'LST', 'doy_sin', 'doy_cos', 'lat_norm']
categorical column: lc_dom
train_us: n=23519, blocks=460
val_us_overlap: n=9723, blocks=379
val_us_nonoverlap: n=661, blocks=63
test_external: n=764, blocks=82
=== US Transfer Temporal dataset ===
window_size: 4
rows: 32051
train_us: n=21578, sites=601
val_us_overlap: n=9530, sites=529
val_us_nonoverlap: n=466, sites=50
test_external: n=477, sites=36
[repeat 1/1] training TemporalCNN_TabNetStatic
[repeat 1] TemporalCNN_TabNetStatic metrics
  train_us           MAE=18.4705  RMSE=26.3359  R=0.7185
  val_us_overlap     MAE=18.8058  RMSE=26.6805  R=0.6592
  val_us_nonoverlap  MAE=24.3361  RMSE=35.0009  R=0.4218
  test_external      MAE=159.4324  RMSE=286.9098  R=0.1307
Saved raw results to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts

In [50]:
# 8. 新 split + Temporal CNN + TabNetStatic + MoE
# 这里运行 1 个模型：TemporalCNN_TabNetStatic_MoE
!python ..\function\lfmc_ml\run_lfmc_us_transfer_temporal_batch.py --repeats 1 --window-size 4 --models "TemporalCNN_TabNetStatic_MoE"


Device: cuda
=== US Transfer S6-lite dataset ===
rows: 34667
numeric columns: ['VOD_Ku_Hpol_Asc', 'VOD_Ku_Vpol_Asc', 'VOD_X_Hpol_Asc', 'VOD_X_Vpol_Asc', 'VOD_C_Hpol_Asc', 'VOD_C_Vpol_Asc', 'LAI', 'Hveg', 'LST', 'doy_sin', 'doy_cos', 'lat_norm']
categorical column: lc_dom
train_us: n=23519, blocks=460
val_us_overlap: n=9723, blocks=379
val_us_nonoverlap: n=661, blocks=63
test_external: n=764, blocks=82
=== US Transfer Temporal dataset ===
window_size: 4
rows: 32051
train_us: n=21578, sites=601
val_us_overlap: n=9530, sites=529
val_us_nonoverlap: n=466, sites=50
test_external: n=477, sites=36
[repeat 1/1] training TemporalCNN_TabNetStatic_MoE
[repeat 1] TemporalCNN_TabNetStatic_MoE metrics
  train_us           MAE=18.9642  RMSE=26.9659  R=0.7051
  val_us_overlap     MAE=18.8372  RMSE=26.7850  R=0.6584
  val_us_nonoverlap  MAE=24.5840  RMSE=35.3699  R=0.4190
  test_external      MAE=143.0228  RMSE=258.2054  R=0.1116
Saved raw results to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\a

In [51]:
# 1. TemporalCNN_ResNetStatic 生成 20200115 的 HDF5 产品
!python ..\function\lfmc_ml\run_lfmc_us_transfer_temporal_prod.py --artifact-dir "..\notebooks\artifacts\dl_us_transfer_temporal\TemporalCNN_ResNetStatic_rep1" --date 20200115 --output-path "G:\data\LFMC\data-TemporalCNN-ResNetStatic\lfmc_pred_20200115.h5" --nonveg-threshold 0.7


[OK] TemporalCNN_ResNetStatic raster prediction finished -> G:\data\LFMC\data-TemporalCNN-ResNetStatic\lfmc_pred_20200115.h5


2026-04-09 16:53:49,983 - INFO - 开始构造 2020-01-12 的产品输入表，共 6,480,000 个像元
2026-04-09 16:53:50,093 - INFO - Hveg 加载成功: Hveg, shape=(1800, 3600)
2026-04-09 16:53:51,963 - INFO - 8日 LAI 可用日期数: 1012
2026-04-09 16:53:56,433 - INFO - LC 加载成功: 2020
2026-04-09 16:54:14,603 - INFO - 开始构造 2020-01-13 的产品输入表，共 6,480,000 个像元
2026-04-09 16:54:37,239 - INFO - 开始构造 2020-01-14 的产品输入表，共 6,480,000 个像元
2026-04-09 16:55:01,702 - INFO - 开始构造 2020-01-15 的产品输入表，共 6,480,000 个像元


In [52]:
# 2. TemporalCNN_ResNetStatic_MoE 生成 20200115 的 HDF5 产品
!python ..\function\lfmc_ml\run_lfmc_us_transfer_temporal_prod.py --artifact-dir "..\notebooks\artifacts\dl_us_transfer_temporal\TemporalCNN_ResNetStatic_MoE_rep1" --date 20200115 --output-path "G:\data\LFMC\data-TemporalCNN-ResNetStatic-MoE\lfmc_pred_20200115.h5" --nonveg-threshold 0.7


[OK] TemporalCNN_ResNetStatic_MoE raster prediction finished -> G:\data\LFMC\data-TemporalCNN-ResNetStatic-MoE\lfmc_pred_20200115.h5


2026-04-09 16:55:40,423 - INFO - 开始构造 2020-01-12 的产品输入表，共 6,480,000 个像元
2026-04-09 16:55:40,569 - INFO - Hveg 加载成功: Hveg, shape=(1800, 3600)
2026-04-09 16:55:42,193 - INFO - 8日 LAI 可用日期数: 1012
2026-04-09 16:55:46,514 - INFO - LC 加载成功: 2020
2026-04-09 16:56:03,706 - INFO - 开始构造 2020-01-13 的产品输入表，共 6,480,000 个像元
2026-04-09 16:56:26,795 - INFO - 开始构造 2020-01-14 的产品输入表，共 6,480,000 个像元
2026-04-09 16:56:50,612 - INFO - 开始构造 2020-01-15 的产品输入表，共 6,480,000 个像元


即使当前最好的两个时序模型，也仍然存在明显的纬度带偏差，说明“泛化指标相对更好”不等于“全球反演图已经合理”。

In [56]:
# 旧 split + Temporal CNN + MoE
!python ..\function\lfmc_ml\run_lfmc_final_temporal_variants_batch.py --repeats 1 --window-size 4 --models "TemporalCNN_MoE"


Device: cuda
=== Final S6-lite Temporal dataset ===
window_size: 4
rows: 32051
train: n=25707, sites=609
val: n=3317, sites=76
test: n=3027, sites=74
[repeat 1/1] training TemporalCNN_MoE
[repeat 1] TemporalCNN_MoE metrics
  train MAE=19.4590  RMSE=27.7425  R=0.6729
  val   MAE=20.4157  RMSE=28.3856  R=0.6428
  test  MAE=18.9675  RMSE=26.4329  R=0.6713
Saved raw results to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl_temporal_variants\lfmc_final_s6_lite_temporal_variants_raw.csv
Saved summary results to: d:\Python\jupyter\jupyter\LFMCRegressor\notebooks\artifacts\dl_temporal_variants\lfmc_final_s6_lite_temporal_variants_summary.csv


In [57]:
# 旧 split + Temporal CNN + MoE
!python ..\function\lfmc_ml\run_lfmc_us_transfer_temporal_prod.py --artifact-dir "..\notebooks\artifacts\dl_temporal_variants\TemporalCNN_MoE_rep1" --date 20200115 --output-path "G:\data\LFMC\data-TemporalCNN-MoE-oldsplit\lfmc_pred_20200115.h5" --nonveg-threshold 0.7


[OK] TemporalCNN_MoE raster prediction finished -> G:\data\LFMC\data-TemporalCNN-MoE-oldsplit\lfmc_pred_20200115.h5


2026-04-09 17:28:06,687 - INFO - 开始构造 2020-01-12 的产品输入表，共 6,480,000 个像元
2026-04-09 17:28:06,804 - INFO - Hveg 加载成功: Hveg, shape=(1800, 3600)
2026-04-09 17:28:08,619 - INFO - 8日 LAI 可用日期数: 1012
2026-04-09 17:28:12,929 - INFO - LC 加载成功: 2020
2026-04-09 17:28:30,192 - INFO - 开始构造 2020-01-13 的产品输入表，共 6,480,000 个像元
2026-04-09 17:28:53,347 - INFO - 开始构造 2020-01-14 的产品输入表，共 6,480,000 个像元
2026-04-09 17:29:16,391 - INFO - 开始构造 2020-01-15 的产品输入表，共 6,480,000 个像元
